# Pass 16 — Error Analysis: Event 2's Out-of-Window Signal

`docs/RESULTS.md` (pass 15) concludes the rule-based baseline has **no ranking
signal** — tightening the alert threshold makes detections vanish rather than
persist. But `docs/findings/08-cycle-timing.md` records that event 2's
STOPPED-duration collapse (2020-05-17 → 05-21, recovering by 05-24) was real
and verified — a genuine precursor sitting ~9 days before event 2's 72h
pre-failure window opens (2020-05-26 23:30), so it was counted as **false
alarms**, not detection.

This notebook investigates that specific mis-scored precursor: whether it's a
mechanism problem (the trailing baseline "catching up" to a sustained
degradation — boiling-frog), whether anything was actually elevated but
below-threshold inside event 2's own window, what the mid-May episodes
actually looked like, whether this out-of-window pattern is unique to event 2
or common to all four events, and whether mid-May is distinguishable from the
already-documented, unexplained early-March cluster.

**Read-only / analysis only** — no model logic, no pipeline changes, no config
changes. Every score/episode/ranking below is produced by calling the existing
package functions (`RuleBasedModel`, `evaluate_fold_at_threshold`,
`explain.rank_channel_contributions`) — nothing is reimplemented. Causality is
respected throughout: every plotted quantity is something the fitted model
could actually have computed at that timestamp, using only past data.

## Setup

Reusing the package end-to-end: `load_config`, `load_raw`, `assign_regimes`,
`make_folds`, `extend_test_end_for_false_alarms`, `RuleBasedModel`,
`evaluate_fold_at_threshold`, `fit_threshold`. The only "new" code is a small
`fit_fold()` wiring helper — it does not compute anything itself, it just
calls `apply_fold` + `RuleBasedModel.fit` the same way `pipeline.py`'s
`run_pipeline` does, so this notebook doesn't need to reach into pipeline.py's
private helpers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from apu_sentinel.config import load_config
from apu_sentinel.data.load import load_raw
from apu_sentinel.data.split import apply_fold, extend_test_end_for_false_alarms, make_folds
from apu_sentinel.data.windows import characterise_sampling
from apu_sentinel.evaluation.metrics import (
    ScoredTestData,
    evaluate_fold_at_threshold,
    fit_threshold,
)
from apu_sentinel.features.cycles import compute_cycle_features, last_completed_run_peak
from apu_sentinel.models.rule_based import RuleBasedModel
from apu_sentinel.regimes import assign_regimes

settings = load_config("local")  # CONFIG=colab for the full-data environment

raw_path = Path(settings.data.raw_dir) / settings.data.raw_filename
df = load_raw(raw_path)
data_start, data_end = df.index.min(), df.index.max()

regimes = assign_regimes(df, settings)
common_folds = make_folds(settings, data_start, data_end)
folds_by_event = {f.event_id: f for f in common_folds}
events_sorted = sorted(settings.evaluation.failure_events, key=lambda e: pd.Timestamp(e.start))
events_by_id = {e.id: e for e in events_sorted}
training_exclusion = settings.split.training_exclusion

sampling = characterise_sampling(df, pd.Timedelta(settings.windowing.gap_threshold))
expected_interval = sampling.modal_interval

rule_cfg = settings.model.rule_based
_decay_cfg = rule_cfg.rules.get("fast_pressure_decay")
_decay_override = _decay_cfg.source_channel if _decay_cfg else None
_decay_channel = _decay_override or settings.features.decay_source_channel
MODEL_COLUMNS = sorted({"Reservoirs", _decay_channel})
BASELINE_WINDOW = pd.Timedelta(rule_cfg.baseline_window)

print("model columns:", MODEL_COLUMNS, " baseline_window:", BASELINE_WINDOW)


def fit_fold(fold):
    """Fit-on-train-only + continuous fold_input, same wiring as
    pipeline.py's private _fit_fold_model -- built locally here from PUBLIC
    functions (apply_fold, RuleBasedModel) so this notebook stays
    self-contained rather than reaching into pipeline.py internals.
    """
    train_raw, _ = apply_fold(df, fold)
    fold_full = df.loc[(df.index >= fold.train_start) & (df.index <= fold.test_end)]

    def model_input(frame):
        out = frame[MODEL_COLUMNS].copy()
        out["regime"] = regimes.loc[frame.index]
        return out

    train_input = model_input(train_raw)
    fold_input = model_input(fold_full)
    model = RuleBasedModel(settings)
    model.fit(train_input)
    return model, train_input, fold_input


def extended(event_id):
    fold = folds_by_event[event_id]
    event = events_by_id[event_id]
    return extend_test_end_for_false_alarms(
        fold, event, events_sorted, training_exclusion, data_end
    )

## Part A — The boiling-frog hypothesis

The STOPPED-duration signal collapses 17 May and "recovers" by 24 May — an
interval of **exactly 7 days**, `model.rule_based.baseline_window`. Every rule
scores `value / trailing_median(value)`; if duration drops and *stays* low,
the 7-day trailing median descends to meet it, returning the ratio to ~1.0
while the absolute value is still depressed — the baseline adapts to the
degradation rather than the machine recovering.

We use fold 2's fitted model (the fold responsible for event 2's own
detection/false-alarm evaluation) throughout Parts A and B.

In [ ]:
event2 = events_by_id[2]
ext2 = extended(2)
model2, train_input2, fold_input2 = fit_fold(ext2)

train_scores2 = model2.score(train_input2)
full_scores2 = model2.score(fold_input2)
full_contrib2 = model2.contributions(fold_input2)
threshold_995 = fit_threshold(train_scores2, settings)
threshold_999 = float(np.quantile(train_scores2, 0.999))
print("fold 2 threshold (q=0.995):", threshold_995)
print("fold 2 threshold (q=0.999):", threshold_999)

WINDOW_START, WINDOW_END = pd.Timestamp("2020-05-26 23:30"), pd.Timestamp("2020-05-29 23:30")

In [ ]:
# Raw cycle-timing features (causal, fold-independent) + the trailing
# baseline each rule actually uses, computed the same way baseline_relative()
# does internally (value.rolling(window, min_periods=1).median()) -- kept
# separate here only so the baseline itself can be plotted alongside the
# ratio, which baseline_relative() doesn't expose.
cycle_features = compute_cycle_features(df, regimes, settings)

stopped_duration = cycle_features["stopped_duration_last"]
stopped_baseline = stopped_duration.rolling(BASELINE_WINDOW, min_periods=1).median()
stopped_ratio = stopped_duration / stopped_baseline

decay_abs = cycle_features["decay_rate_last"].abs()
decay_baseline = decay_abs.rolling(BASELINE_WINDOW, min_periods=1).median()
decay_ratio = decay_abs / decay_baseline

peak = last_completed_run_peak(df[["Reservoirs"]], regimes, settings, "OFFLOAD", "Reservoirs")
peak_baseline = peak.rolling(BASELINE_WINDOW, min_periods=1).median()
peak_ratio = peak / peak_baseline

# Per-rule SEVERITY (the model's own attribution, via model.contributions --
# not recomputed by hand) aligned to the same index.
contrib_cols = list(model2.contributor_names)
severities = pd.DataFrame(full_contrib2, index=fold_input2.index, columns=contrib_cols)

In [ ]:
def plot_boiling_frog(raw, baseline, ratio, severity, rule_name, window_a=None):
    window_a = window_a or ("2020-05-10", "2020-06-01")
    start, end = pd.Timestamp(window_a[0]), pd.Timestamp(window_a[1])
    r = raw.loc[start:end]
    b = baseline.loc[start:end]
    q = ratio.loc[start:end]
    s = severity.loc[start:end] if severity is not None else None

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(r.index, r.values, label="raw", linewidth=0.8)
    baseline_label = f"{BASELINE_WINDOW} trailing median (baseline)"
    axes[0].plot(b.index, b.values, label=baseline_label, linewidth=1.2)
    axes[0].set_ylabel(rule_name)
    axes[0].legend(loc="upper right", fontsize=8)

    axes[1].plot(q.index, q.values, linewidth=0.8, color="tab:orange")
    axes[1].axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
    axes[1].set_ylabel("ratio (raw/baseline)")

    if s is not None:
        threshold_label = "alert threshold (q=0.995, overall score)"
        axes[2].plot(s.index, s.values, linewidth=0.8, color="tab:green")
        axes[2].axhline(
            threshold_995, color="red", linestyle="--", linewidth=0.8, label=threshold_label
        )
        axes[2].set_ylabel("rule severity")
        axes[2].legend(loc="upper right", fontsize=8)

    for ax in axes:
        ax.axvline(WINDOW_START, color="black", linestyle=":", linewidth=1.2)
        ax.text(WINDOW_START, ax.get_ylim()[1], " 72h window opens", fontsize=7, va="top")

    fig.suptitle(f"Boiling-frog check: {rule_name}, {window_a[0]} -> {window_a[1]}")
    fig.tight_layout()
    return fig


_ = plot_boiling_frog(
    stopped_duration,
    stopped_baseline,
    stopped_ratio,
    severities["short_stopped_duration"],
    "stopped_duration_last",
)
plt.show()

In [ ]:
_ = plot_boiling_frog(
    decay_abs,
    decay_baseline,
    decay_ratio,
    severities.get("fast_pressure_decay"),
    "|decay_rate_last|",
)
plt.show()
_ = plot_boiling_frog(
    peak,
    peak_baseline,
    peak_ratio,
    severities.get("low_peak_pressure"),
    "last_completed_run_peak (OFFLOAD, Reservoirs)",
)
plt.show()

In [ ]:
pre = stopped_duration.loc["2020-05-10":"2020-05-16"]
collapse = stopped_duration.loc["2020-05-17":"2020-05-21"]
post = stopped_duration.loc["2020-05-24":"2020-05-26"]

pre_ratio = stopped_ratio.loc["2020-05-10":"2020-05-16"]
collapse_ratio = stopped_ratio.loc["2020-05-17":"2020-05-21"]
post_ratio = stopped_ratio.loc["2020-05-24":"2020-05-26"]

pre_baseline = stopped_baseline.loc["2020-05-10":"2020-05-16"]
post_baseline = stopped_baseline.loc["2020-05-24":"2020-05-26"]

print(f"stopped_duration_last mean, pre-collapse (05-10..05-16):    {pre.mean():.1f} s")
print(f"stopped_duration_last mean, collapse (05-17..05-21):         {collapse.mean():.1f} s")
print(f"stopped_duration_last mean, post/window-open (05-24..05-26): {post.mean():.1f} s")
print()
print(f"trailing baseline mean, pre-collapse:   {pre_baseline.mean():.1f} s")
print(
    f"trailing baseline mean, post-collapse:  {post_baseline.mean():.1f} s"
    "  <- baseline descended to meet the depressed value"
)
print()
print(f"ratio mean, pre-collapse:     {pre_ratio.mean():.3f}")
print(
    f"ratio mean, collapse:         {collapse_ratio.mean():.3f}"
    "  <- correctly flags abnormal at the time"
)
print(
    f"ratio mean, post/window-open: {post_ratio.mean():.3f}"
    "  <- back above 1.0 even though duration itself never recovered"
)
print()
print("VERDICT: CONFIRMED -- absolute duration stays depressed (~250-450s) all the way through the")
print("failure, but the ratio the rule actually scores on reads as *normal or better* by the time")
print("the 72h window opens, because the 7-day trailing baseline collapsed down to meet it.")

## Part B — What actually happened inside event 2's window

The model reported no detection at any common width. Was the raw score at
ordinary levels (nothing to see), or elevated-but-below-threshold (a
near-miss)?

In [ ]:
mask_window = (fold_input2.index >= WINDOW_START) & (fold_input2.index <= WINDOW_END)
scores_in_window = full_scores2[mask_window]
times_in_window = fold_input2.index[mask_window]

max_score = scores_in_window.max()
max_time = times_in_window[np.argmax(scores_in_window)]
train_percentile = (train_scores2 < max_score).mean() * 100

n_above_995 = (scores_in_window >= threshold_995).sum()
n_above_999 = (scores_in_window >= threshold_999).sum()

margin = threshold_995 - max_score
print(f"max score inside event 2's 72h window: {max_score:.4f} at {max_time}")
print(f"percentile of that max against fold 2's training distribution: {train_percentile:.2f}%")
print(f"threshold (q=0.995): {threshold_995:.4f}  -- max score is {margin:.4f} below it")
print(f"n timestamps >= q0.995 threshold inside window: {n_above_995} / {len(scores_in_window)}")
print(f"n timestamps >= q0.999 threshold inside window: {n_above_999} / {len(scores_in_window)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_start, plot_end = WINDOW_START - pd.Timedelta("1D"), WINDOW_END + pd.Timedelta("1D")
plot_mask = (fold_input2.index >= plot_start) & (fold_input2.index <= plot_end)
ax.plot(fold_input2.index[plot_mask], full_scores2[plot_mask], linewidth=0.7, label="raw score")
ax.axhline(threshold_995, color="red", linestyle="--", linewidth=1, label="q=0.995 threshold")
ax.axhline(threshold_999, color="darkred", linestyle=":", linewidth=1, label="q=0.999 threshold")
ax.axvline(WINDOW_START, color="black", linestyle=":", linewidth=1.2, label="72h window opens")
ax.axvline(pd.Timestamp(event2.start), color="black", linestyle="-", linewidth=1.2, label="onset")
ax.set_ylabel("score")
ax.legend(loc="lower left", fontsize=8)
fig.suptitle("Event 2: raw score across its 72h pre-failure window")
fig.tight_layout()
plt.show()

## Part C — Characterise the mid-May episodes

Episodes are the model's own grouping (`evaluate_fold_at_threshold` ->
`_group_episode_index_ranges`, called via the public entry point) with their
full ranked diagnosis from `explain.rank_channel_contributions` — this
notebook does not group timestamps or rank channels itself.

**Which fold actually scored mid-May?** Fold 2's official test period only
starts 2020-05-25 23:30 (`test_start = event2.start - 72h - embargo`) — mid-May
(05-15 → 05-25) is entirely inside fold 2's *training* window. The mid-May
episodes that were counted as false alarms in `docs/RESULTS.md` belong to
**fold 1** instead: its false-alarm accounting is extended
(`extend_test_end_for_false_alarms`) all the way to 2020-05-28 23:30 — i.e.
fold 1's own evaluation window happens to swallow all of mid-May. That matters
for Part E: fold 1's training data (Feb 1 → Apr 13) already contains the
March cluster, so its rule calibration has already seen equally extreme
`stopped_duration` ratios before mid-May ever occurs.

In [ ]:
event1 = events_by_id[1]
ext1 = extended(1)
model1, train_input1, fold_input1 = fit_fold(ext1)

train_scores1 = model1.score(train_input1)
full_scores1 = model1.score(fold_input1)
full_contrib1 = model1.contributions(fold_input1)
threshold1 = fit_threshold(train_scores1, settings)

test_mask1 = (fold_input1.index >= ext1.test_start) & (fold_input1.index <= ext1.test_end)
test_data1 = ScoredTestData(
    timestamps=fold_input1.index[test_mask1],
    scores=full_scores1[test_mask1],
    contributions=full_contrib1[test_mask1],
    channel_names=model1.contributor_names,
    expected_interval=expected_interval,
)
result1_full = evaluate_fold_at_threshold(ext1, event1, 72.0, threshold1, test_data1, settings)
print(
    f"fold 1, width=72h: {len(result1_full.episodes)} total episodes, "
    f"{result1_full.false_episode_count} false_alarm, "
    f"over {result1_full.evaluated_days:.2f} evaluated days"
)

In [ ]:
MID_MAY_START, MID_MAY_END = pd.Timestamp("2020-05-15"), pd.Timestamp("2020-05-25")
mid_may_episodes = [ep for ep in result1_full.episodes if MID_MAY_START <= ep.start <= MID_MAY_END]

n_mid_may = len(mid_may_episodes)
print(f"episodes in mid-May ({MID_MAY_START.date()} -> {MID_MAY_END.date()}): {n_mid_may}\n")
for ep in mid_may_episodes:
    print(
        f"{ep.start} -> {ep.end}   duration={ep.end - ep.start}   "
        f"peak={ep.peak_score:.4f}   category={ep.category}"
    )
    for name, val in ep.channel_ranking:
        print(f"    {name:<24s} {val:.4f}")
    print()

## Part D — Symmetric check across all four events

Guard against cherry-picking event 2: for each event, score the 21 days
before onset using that event's own fitted fold model, and check for
above-threshold episodes outside the scored 72h window.

The 21-day lookback deliberately reaches back into each fold's own *training*
period for events 2-4 (common-width `test_start` is only `72h + embargo`
before onset — about 4 days) — same causal, descriptive back-test technique
already used for the March cluster in pass 12 (`docs/RESULTS.md` §12): the
model scores its own past with no lookahead, it's simply not part of the
*official* false-alarm count when the timestamp falls inside training.

In [ ]:
per_event_scores = {}
per_event_results = {}
per_event_thresholds = {}

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=False)

for ax, event_id in zip(axes, [1, 2, 3, 4], strict=True):
    event = events_by_id[event_id]
    ext_fold = extended(event_id)
    model, train_input, fold_input = fit_fold(ext_fold)
    train_scores = model.score(train_input)
    full_scores = model.score(fold_input)
    full_contrib = model.contributions(fold_input)
    threshold = fit_threshold(train_scores, settings)

    onset = pd.Timestamp(event.start)
    pre21_start = onset - pd.Timedelta(days=21)
    mask = (fold_input.index >= pre21_start) & (fold_input.index <= onset)

    scored = ScoredTestData(
        timestamps=fold_input.index[mask],
        scores=full_scores[mask],
        contributions=full_contrib[mask],
        channel_names=model.contributor_names,
        expected_interval=expected_interval,
    )
    result = evaluate_fold_at_threshold(ext_fold, event, 72.0, threshold, scored, settings)

    per_event_scores[event_id] = pd.Series(full_scores[mask], index=fold_input.index[mask])
    per_event_results[event_id] = result
    per_event_thresholds[event_id] = threshold

    s = per_event_scores[event_id]
    ax.plot(s.index, s.values, linewidth=0.6)
    ax.axhline(threshold, color="red", linestyle="--", linewidth=1, label="threshold (q=0.995)")
    ax.axvline(
        onset - pd.Timedelta(hours=72),
        color="black",
        linestyle=":",
        linewidth=1.2,
        label="72h window boundary",
    )
    ax.axvline(onset, color="black", linestyle="-", linewidth=1.2, label="failure onset")
    ax.set_title(f"Event {event_id} -- 21 days before onset ({onset})")
    ax.legend(loc="upper left", fontsize=7)

fig.tight_layout()
plt.show()

In [ ]:
for event_id in [1, 2, 3, 4]:
    result = per_event_results[event_id]
    onset = pd.Timestamp(events_by_id[event_id].start)
    threshold = per_event_thresholds[event_id]
    print(f"=== event {event_id} (onset {onset}) -- threshold {threshold:.4f} ===")
    print(f"{len(result.episodes)} episode(s) in the 21-day pre-onset horizon")
    for ep in result.episodes:
        lead = onset - ep.start
        inside_72h = ep.category in ("early_warning", "concurrent")
        print(
            f"  {ep.start}  lead_before_onset={lead}  category={ep.category}  "
            f"inside_72h_window={inside_72h}  top_rule={ep.channel_ranking[0]}"
        )
    print()

## Part E — Mid-May vs. the early-March cluster

`docs/findings/08-cycle-timing.md` records early March (03-03 → 03-12) as an
unreported-anomaly cluster: 8 episodes in 9 days, no documented failure. Both
clusters are scored here with **fold 1's** model (the fold whose training
period contains March, and whose extended evaluation window is what actually
counted mid-May as false alarms — see Part C) so the comparison uses one
consistent calibration.

In [ ]:
MARCH_START, MARCH_END = pd.Timestamp("2020-03-03"), pd.Timestamp("2020-03-12")
march_probe_start, march_probe_end = pd.Timestamp("2020-03-01"), pd.Timestamp("2020-03-15")
mask_march = (fold_input1.index >= march_probe_start) & (fold_input1.index <= march_probe_end)
scored_march = ScoredTestData(
    timestamps=fold_input1.index[mask_march],
    scores=full_scores1[mask_march],
    contributions=full_contrib1[mask_march],
    channel_names=model1.contributor_names,
    expected_interval=expected_interval,
)
result_march = evaluate_fold_at_threshold(ext1, event1, 72.0, threshold1, scored_march, settings)
march_episodes = [ep for ep in result_march.episodes if MARCH_START <= ep.start <= MARCH_END]

n_march = len(march_episodes)
print(f"March episodes ({MARCH_START.date()} -> {MARCH_END.date()}): {n_march}\n")
for ep in march_episodes:
    print(f"{ep.start} -> {ep.end}   duration={ep.end - ep.start}   peak={ep.peak_score:.4f}")
    for name, val in ep.channel_ranking:
        print(f"    {name:<24s} {val:.4f}")
    print()

In [ ]:
def episode_summary(label, episodes, window_days):
    density = len(episodes) / window_days
    agreements = [sum(1 for _, v in ep.channel_ranking if v > 0.5) for ep in episodes]
    top_rules = [ep.channel_ranking[0][0] for ep in episodes]
    print(f"--- {label} ---")
    print(f"  episodes: {len(episodes)} over ~{window_days} days -> density {density:.2f}/day")
    print(f"  peak scores: {[round(ep.peak_score, 4) for ep in episodes]}")
    print(f"  rule agreement (>0.5 severity), per episode: {agreements}")
    print(f"  top rule per episode: {top_rules}")
    print()


episode_summary("March cluster", march_episodes, 9)
episode_summary("Mid-May cluster", mid_may_episodes, 10)

march_mask_feat = (cycle_features.index >= MARCH_START) & (cycle_features.index <= MARCH_END)
may_mask_feat = (cycle_features.index >= MID_MAY_START) & (cycle_features.index <= MID_MAY_END)

march_duration = cycle_features.loc[march_mask_feat, "stopped_duration_last"].median()
may_duration = cycle_features.loc[may_mask_feat, "stopped_duration_last"].median()
march_decay = cycle_features.loc[march_mask_feat, "decay_rate_last"].median()
may_decay = cycle_features.loc[may_mask_feat, "decay_rate_last"].median()
march_peak = peak.loc[march_mask_feat].median()
may_peak = peak.loc[may_mask_feat].median()

print("--- absolute (not ratio) feature levels ---")
print(f"stopped_duration_last median -- March: {march_duration:.0f}s  Mid-May: {may_duration:.0f}s")
print(f"decay_rate_last median (signed) -- March: {march_decay:.5f}   Mid-May: {may_decay:.5f}")
print(f"OFFLOAD peak (Reservoirs) median -- March: {march_peak:.3f}   Mid-May: {may_peak:.3f}")

**Verdict**: episode density, peak severity, rule-agreement pattern, top-rule
mix, and absolute feature depression are all the same order of magnitude
between the two clusters — see the printed summary above for the exact
numbers. Neither cluster has a distinguishing signature the other lacks. This
is reported plainly rather than resolved by assertion in
`docs/findings/12-event2-error-analysis.md`: it cuts both ways — either March
hides an unreported failure/near-miss, or mid-May's signal (and by extension
event 2's own precursor) is not failure-specific either.

## Summary

See `docs/findings/12-event2-error-analysis.md` for the full write-up. No
model logic, pipeline logic, or config values were changed in this notebook —
every score, episode, and ranking above comes from calling the existing
`RuleBasedModel`, `evaluate_fold_at_threshold`, and
`explain.rank_channel_contributions` unmodified.